# Verify GPU

In [1]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("Device Name:", torch.cuda.get_device_name(0))

    print("CUDA Version:", torch.version.cuda)

    print(
        "Current Device:",
        torch.cuda.current_device()
    )

    print(
        "Device Count:",
        torch.cuda.device_count()
    )

    x = torch.rand(3, 3).cuda()

    print("\nTensor Device:", x.device)

else:

    print("Running on CPU")

CUDA Available: False
Running on CPU


In [1]:
pip install optuna

Looking in indexes: https://pypi.org/simple, https://pip.repos.neuron.amazonaws.com
  Obtaining dependency information for optuna from https://files.pythonhosted.org/packages/ab/f3/e5fcd5d9b15771ed6dc10e3a7eeddc672e418f4f4c4653d216cc1d857e2d/optuna-4.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for alembic>=1.5.0 from https://files.pythonhosted.org/packages/96/78/5fe6dc3a3a5b2f5a2a4faef8bfe336d5fa049a38884ab3172e0098160c01/alembic-1.18.5-py3-none-any.whl.metadata
  Obtaining dependency information for colorlog from https://files.pythonhosted.org/packages/6d/c1/e419ef3723a074172b68aaa89c9f3de486ed4c2399e2dbd8113a4fdcaf9e/colorlog-6.10.1-py3-none-any.whl.metadata
  Obtaining dependency information for sqlalchemy>=1.4.2 from https://files.pythonhosted.org/packages/2b/7c/7ab9f9aadc5944fdd06612484ed7918fe376ad871a5f50404dc1536e0194/sqlalchemy-2.0.51-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata
  Obtaining dependency informatio

# Hyperparameter Search

In [5]:
"""
Hybrid ACO + RL-GNN for TSP
============================
Replaces tabular Q-learning with a Graph Attention Network (GAT) policy.

Architecture:
  - GNN encoder:  2-layer GAT over fully-connected city graph
                  -> node embeddings h_i in R^{hidden_dim}
  - Policy head:  attention between current-city query and all unvisited keys
                  -> probability distribution over next city
  - Training:     REINFORCE with baseline (mean tour length of the ant batch)
  - ACO wrapper:  pheromone matrix + 2-opt local search unchanged from original

Jupyter/notebook safe: no multiprocessing, runs sequentially on GPU or CPU.

Dependencies:
    pip install torch torch-geometric numpy
    (torch-geometric wheels: https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html)
"""

import os
import csv
import random
import math
import optuna
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data


# =============================================================================
# 1.  TSP file reader
# =============================================================================

def read(instance_file):
    coords = []
    in_coord_section = False
    with open(instance_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("NODE_COORD_SECTION"):
                in_coord_section = True
                continue
            if not in_coord_section:
                continue
            if line.startswith("EOF"):
                break
            parts = line.split()
            if len(parts) < 3:
                continue
            _, x, y = parts[:3]
            coords.append([float(x), float(y)])
    return np.array(coords)


# =============================================================================
# 2.  Distance / path utilities
# =============================================================================

def calculate_path_length(path, distances):
    return sum(distances[path[i]][path[i + 1]] for i in range(len(path) - 1))


def local_search_2opt(path, distances, num_cities):
    best = path[:]
    best_len = calculate_path_length(best, distances)
    improved = True
    while improved:
        improved = False
        for i in range(1, num_cities - 2):
            for j in range(i + 1, num_cities):
                if j - i == 1:
                    continue
                candidate = path[:i] + path[i:j][::-1] + path[j:]
                clen = calculate_path_length(candidate, distances)
                if clen < best_len:
                    best, best_len = candidate, clen
                    improved = True
        path = best
    return best


# =============================================================================
# 3.  GNN Policy Network  (Graph Attention Network backbone)
# =============================================================================

class GNNPolicyNet(nn.Module):
    """
    Graph Attention Network that maps city coordinates to a next-city
    probability distribution conditioned on the current city.

    Input features per node (dim=5):
        [x_norm, y_norm, dist_to_centroid_norm, sin(angle), cos(angle)]

    Architecture:
        Linear(5 -> hidden_dim)
        GATConv(hidden_dim -> hidden_dim, heads=num_heads, concat) x num_layers
        Policy head: score(i) = (W_q * h_current) . (W_k * h_i) / sqrt(D)
    """

    def __init__(self, hidden_dim: int = 128, num_heads: int = 4, num_layers: int = 2):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Node feature encoder
        self.input_proj = nn.Linear(5, hidden_dim)

        # GAT layers: each head outputs hidden_dim//num_heads, concat -> hidden_dim
        head_dim = hidden_dim // num_heads
        self.gat_layers = nn.ModuleList([
            GATConv(hidden_dim, head_dim, heads=num_heads, concat=True, dropout=0.0)
            for _ in range(num_layers)
        ])

        # Attention-based policy head
        self.W_query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_key   = nn.Linear(hidden_dim, hidden_dim, bias=False)

        # Xavier initialisation
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def encode(self, data: Data) -> torch.Tensor:
        """Run GAT message passing, return node embeddings [N, hidden_dim]."""
        h = F.relu(self.input_proj(data.x))
        for gat in self.gat_layers:
            h = F.elu(gat(h, data.edge_index))
        return h

    def policy(
        self,
        node_emb: torch.Tensor,     # [N, hidden_dim]
        current_idx: int,
        visited_mask: torch.Tensor  # bool [N], True = already visited
    ) -> torch.Tensor:
        """Return log-probabilities over cities; visited cities -> -inf."""
        query  = self.W_query(node_emb[current_idx]).unsqueeze(0)  # [1, D]
        keys   = self.W_key(node_emb)                              # [N, D]
        scale  = math.sqrt(self.hidden_dim)
        logits = (query @ keys.T).squeeze(0) / scale               # [N]
        logits = logits.masked_fill(visited_mask, float('-inf'))
        return F.log_softmax(logits, dim=-1)

    def forward(self, data, current_idx, visited_mask):
        return self.policy(self.encode(data), current_idx, visited_mask)


# =============================================================================
# 4.  Graph construction helper
# =============================================================================

def build_graph(coords: np.ndarray, device: torch.device) -> Data:
    """
    Fully-connected PyG Data object from city coordinates.
    Node features: [x_norm, y_norm, dist_centroid_norm, sin_theta, cos_theta]
    """
    N = len(coords)

    lo, hi = coords.min(0), coords.max(0)
    span = (hi - lo).clip(min=1e-6)
    coords_n = (coords - lo) / span

    centroid = coords_n.mean(0)
    diffs    = coords_n - centroid
    dist_c   = np.linalg.norm(diffs, axis=1, keepdims=True)
    max_d    = dist_c.max() + 1e-6
    angles   = np.arctan2(diffs[:, 1], diffs[:, 0])

    node_feats = np.concatenate([
        coords_n,
        dist_c / max_d,
        np.sin(angles)[:, None],
        np.cos(angles)[:, None],
    ], axis=1).astype(np.float32)

    # Fully-connected edges (no self-loops)
    src = [i for i in range(N) for j in range(N) if i != j]
    dst = [j for i in range(N) for j in range(N) if i != j]

    return Data(
        x          = torch.tensor(node_feats, device=device),
        edge_index = torch.tensor([src, dst], dtype=torch.long, device=device),
    )


# =============================================================================
# 5.  ACO pheromone update
# =============================================================================

def update_pheromones(pheromones, paths, distances, w_reward, rho, num_cities):
    delta = np.zeros((num_cities, num_cities))
    for path in paths:
        reward = w_reward / calculate_path_length(path, distances)
        for i in range(len(path) - 1):
            delta[path[i]][path[i + 1]] += reward
    pheromones += -rho * pheromones + delta


# =============================================================================
# 6.  Hybrid ACO + RL-GNN main loop
# =============================================================================

def hybrid_aco_rl_gnn(
    coords: np.ndarray,
    distances: np.ndarray,
    policy_net: GNNPolicyNet,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    # ACO params
    num_episodes: int  = 100,
    num_ants: int      = 20,
    rho: float         = 0.3,
    w_reward: float    = 10.0,
    delta: float       = 3.0,
    beta: float        = 3.0,
    # 2-opt
    num_local_search_iterations: int = 25,
    # RL params
    entropy_coef: float = 0.01,
    # GNN vs pheromone blend  (0 = pure ACO, 1 = pure GNN)
    gnn_weight: float  = 0.6,
    # Logging
    trial_id: int      = 0,
    print_every: int   = 10,
    
    # Early stopping
    patience: int      = 20,
    ):
    """
    Each episode:
      1. Encode graph with GAT -> node embeddings (no grad, rollout only).
      2. Each ant constructs a tour by blending GNN log-probs + ACO desirability.
      3. Recompute log-probs WITH grad, run REINFORCE + entropy regularisation.
      4. 2-opt refinement on best ant tour.
      5. Pheromone update on best tour.
    """
    num_cities = len(coords)
    graph_data = build_graph(coords, device)
    pheromones = np.ones((num_cities, num_cities))

    overall_best_path = None
    overall_best_len  = float('inf')

    #early_patience critertion
    episodes_without_improvement = 0
    
    for episode in range(num_episodes):
        policy_net.train()

        # -- Encode graph once per episode (no grad needed for rollout) -------
        with torch.no_grad():
            node_emb = policy_net.encode(graph_data)  # [N, D]

        paths         = []
        entropy_sums  = []

        # -- Ant rollouts ------------------------------------------------------
        for ant in range(num_ants):
            start_city = random.randint(0, num_cities - 1)
            path       = [start_city]
            visited    = torch.zeros(num_cities, dtype=torch.bool, device=device)
            visited[start_city] = True

            ant_entropies = []

            for step in range(num_cities - 1):
                current = path[-1]

                # GNN log-probabilities
                with torch.no_grad():
                    log_probs_gnn = policy_net.policy(node_emb, current, visited)

                # ACO desirability scores (pheromone x heuristic)
                aco_scores = np.zeros(num_cities)
                for j in range(num_cities):
                    if not visited[j].item():
                        aco_scores[j] = (
                            pheromones[current][j] ** delta *
                            (1.0 / (distances[current][j] + 1e-12)) ** beta
                        )
                aco_sum = aco_scores.sum()
                if aco_sum > 0:
                    aco_scores /= aco_sum

                aco_log = torch.tensor(
                    np.log(aco_scores + 1e-12), dtype=torch.float32, device=device
                ).masked_fill(visited, float('-inf'))

                # Blend in log space and sample
                combined = (
                    gnn_weight * log_probs_gnn +
                    (1 - gnn_weight) * F.log_softmax(aco_log, dim=-1)
                ).masked_fill(visited, float('-inf'))

                probs     = F.softmax(combined, dim=-1)
                dist_obj  = torch.distributions.Categorical(probs=probs)
                next_city = dist_obj.sample().item()

                ant_entropies.append(dist_obj.entropy())
                path.append(next_city)
                visited[next_city] = True

            path.append(path[0])  # close tour
            paths.append(path)
            entropy_sums.append(torch.stack(ant_entropies).mean())

        # -- Tour lengths ------------------------------------------------------
        tour_lengths = np.array([calculate_path_length(p, distances) for p in paths])

        # -- REINFORCE with mean baseline -------------------------------------
        baseline = tour_lengths.mean()
        advantages = torch.tensor(
            -(tour_lengths - baseline), dtype=torch.float32, device=device
        )

        policy_loss = torch.zeros(1, device=device).squeeze()  # scalar, grad-safe initializer
        
        for i in range(num_ants):
            log_probs_list = []
            visited_set = {paths[i][0]}               # plain Python set — no autograd involvement
            node_emb_bp = policy_net.encode(graph_data)  # fresh encoding WITH grad
        
            for step in range(num_cities - 1):
                cur = paths[i][step]
                nxt = paths[i][step + 1]
        
                # Build a brand-new mask tensor each step — never mutates a tracked tensor
                mask = torch.zeros(num_cities, dtype=torch.bool, device=device)
                for v in visited_set:
                    mask[v] = True
        
                lp = policy_net.policy(node_emb_bp, cur, mask)
                log_probs_list.append(lp[nxt])
                visited_set.add(nxt)                  # update set, not any tensor
        
            policy_loss = policy_loss - advantages[i] * torch.stack(log_probs_list).sum()
        
        policy_loss  = policy_loss / num_ants
        entropy_loss = -entropy_coef * torch.stack(entropy_sums).mean()
        total_loss   = policy_loss + entropy_loss

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_net.parameters(), max_norm=1.0)
        optimizer.step()

        # -- 2-opt on best ant ------------------------------------------------
        best_idx  = int(np.argmin(tour_lengths))
        best_path = paths[best_idx]
        best_len  = tour_lengths[best_idx]

        for _ in range(num_local_search_iterations):
            refined     = local_search_2opt(best_path, distances, num_cities)
            refined_len = calculate_path_length(refined, distances)
            if refined_len < best_len:
                best_path, best_len = refined, refined_len

        if best_len < overall_best_len:
            overall_best_len = best_len
            overall_best_path = best_path[:]
            episodes_without_improvement = 0
        else:
            episodes_without_improvement += 1

        # -- Pheromone update -------------------------------------------------
        update_pheromones(pheromones, [best_path], distances, w_reward, rho, num_cities)

        # Early stopping
        if episodes_without_improvement >= patience:
            print(
                f"  [T{trial_id:02d}] Early stopping at episode "
                f"{episode + 1} (no improvement for {patience} episodes)"
            )
            break

        # -- Episode progress print -------------------------------------------
        freq = max(print_every, 1)
        if (episode + 1) % freq == 0 or episode == 0:
            star = " ★" if best_len == overall_best_len else ""
            print(
                f"  [T{trial_id:02d}] Ep {episode+1:4d}/{num_episodes}"
                f"  best={best_len:10.2f}"
                f"  overall={overall_best_len:10.2f}"
                f"  loss={total_loss.item():8.4f}"
                f"  ant_avg={tour_lengths.mean():10.2f}"
                f"  ant_std={tour_lengths.std():7.2f}"
                f"{star}",
                flush=True,
            )

    return overall_best_path


# =============================================================================
# 7.  Trial runner
# =============================================================================

def run_trial(trial, coords, distances, params):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    policy_net = GNNPolicyNet(
        hidden_dim = params["hidden_dim"],
        num_heads  = params["num_heads"],
        num_layers = params["num_layers"],
    ).to(device)

    optimizer = torch.optim.Adam(
        policy_net.parameters(),
        lr           = params["lr"],
        weight_decay = 1e-5,
    )

    print(f"  [Trial {trial:02d}] Starting on {device}")
    start = datetime.now()

    best_path = hybrid_aco_rl_gnn(
        coords       = coords,
        distances    = distances,
        policy_net   = policy_net,
        optimizer    = optimizer,
        device       = device,
        num_episodes = params["num_episodes"],
        num_ants     = params["num_ants"],
        rho          = params["rho"],
        w_reward     = params["w_reward"],
        delta        = params["delta"],
        beta         = params["beta"],
        num_local_search_iterations = params["num_local_search_iterations"],
        entropy_coef = params["entropy_coef"],
        gnn_weight   = params["gnn_weight"],
        patience     = params["patience"],
        trial_id     = trial,
        print_every  = params["print_every"],
    )

    length  = calculate_path_length(best_path, distances)
    elapsed = (datetime.now() - start).total_seconds()
    print(f"  [Trial {trial:02d}] DONE  length={length:.2f}  time={elapsed:.1f}s")
    return trial, length, elapsed


# =============================================================================
# 8.  Hyperparameters
# =============================================================================

global_params = {
    # GNN architecture
    "hidden_dim": 128,
    "num_heads": 2,
    "num_layers": 4,
    "lr": 0.000540030125825684,

    # ACO
    "num_ants": 37,
    "num_episodes": 100,
    "rho": 0.17848728876955464,
    "w_reward": 10.0,
    "delta": 2.2721263607571762,
    "beta": 2.9664340523909023,

    # Stage 2 parameters
    "num_local_search_iterations": 19,
    "patience": 29,

    # These are not used yet
    "epsilon": 0.2,
    "epsilon_decay": 0.995,
    "epsilon_min": 0.05,

    # RL
    "entropy_coef": 0.001559946051309167,

    # Blend
    "gnn_weight": 0.3267091672644127,

    # Logging
    "print_every": 10,
}

def objective(trial, coords, distances):

    params = global_params.copy()

    params["num_episodes"] = trial.suggest_categorical(
        "num_episodes",
        [30, 50, 75, 100, 150, 200]
    )

    lengths = []

    for i in range(3):
        _, length, _ = run_trial(i, coords, distances, params)
        lengths.append(length)

    return sum(lengths) / len(lengths)
# =============================================================================
# 9.  Main — sequential trials, Jupyter/notebook safe
#
#     No multiprocessing: CUDA + fork breaks in Jupyter, and sequential
#     execution is correct on a single GPU anyway (the GPU is already fully
#     utilised per trial by the GNN forward/backward pass).
# =============================================================================

instances_dir  = "SmallInstances"
instance_files = [
    os.path.join(instances_dir, f)
    for f in os.listdir(instances_dir)
    if f.endswith(".tsp")
]

os.makedirs("Results", exist_ok=True)
csv_file   = "Results/ACO-RLGNN-2OptAlgorithm.csv"
num_trials = 5
algorithm  = "ACO-RLGNN-2OptAlgorithm"

with open(csv_file, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Instance", "Algorithm", "Trial", "PathLength", "TimeSeconds"])

    for instance_file in instance_files:
        instance_name = os.path.basename(instance_file).replace(".tsp", "")

        print(f"\n{'='*52}")
        print(f"  Instance : {instance_name}")
        print(f"  Trials   : {num_trials}  |  "
              f"Episodes : {global_params['num_episodes']}  |  "
              f"Ants : {global_params['num_ants']}")
        print(f"{'='*52}")

        coords     = read(instance_file)
        num_cities = len(coords)
        distances  = np.sqrt(
            (coords[:, None, 0] - coords[None, :, 0]) ** 2 +
            (coords[:, None, 1] - coords[None, :, 1]) ** 2
        )

        study = optuna.create_study(direction="minimize")
        
        study.optimize(
            lambda trial: objective(trial, coords, distances),
            n_trials=20
        )
        
        print(f"\n{'='*52}")
        print(f"  {instance_name} — Optuna complete")
        print(f"{'='*52}")
        print(f"Best Path Length : {study.best_value:.2f}")
        print("\nBest Hyperparameters:")
        
        for key, value in study.best_params.items():
            print(f"{key}: {value}")
        
        best_params = global_params.copy()
        best_params.update(study.best_params)
        
        print("\nRunning final evaluation with best hyperparameters...\n")
        
        trial_num, length, tsec = run_trial(
            999,
            coords,
            distances,
            best_params
        )
        
        writer.writerow([
            instance_name,
            algorithm,
            "Best",
            length,
            tsec
        ])
        
        f.flush()
        
        print(f"\nFinal Best Length : {length:.2f}")
        print(f"Final Runtime     : {tsec:.2f} sec")
        print(f"{'='*52}")
        
        print(f"\nAll done. Results saved to: {csv_file}")

[I 2026-06-26 13:59:36,282] A new study created in memory with name: no-name-2564eb0b-4654-4257-8a37-ba3b04461b3f



  Instance : eil51
  Trials   : 5  |  Episodes : 100  |  Ants : 37
  [Trial 00] Starting on cpu
  [T00] Ep    1/75  best=    467.47  overall=    467.47  loss= -0.0036  ant_avg=   1025.28  ant_std=  95.97 ★
  [T00] Ep   10/75  best=    461.83  overall=    456.41  loss= -0.0033  ant_avg=    971.59  ant_std=  83.79
  [T00] Ep   20/75  best=    473.78  overall=    445.81  loss= -0.0025  ant_avg=    768.36  ant_std=  70.41
  [T00] Ep   30/75  best=    442.78  overall=    442.78  loss= -0.0011  ant_avg=    572.52  ant_std=  51.68 ★
  [T00] Ep   40/75  best=    430.45  overall=    430.45  loss= -0.0001  ant_avg=    470.85  ant_std=  46.03
  [T00] Ep   50/75  best=    430.45  overall=    430.45  loss=  0.0001  ant_avg=    436.21  ant_std=  19.59 ★
  [T00] Ep   60/75  best=    430.45  overall=    430.45  loss=  0.0000  ant_avg=    430.68  ant_std=   1.43 ★
  [T00] Early stopping at episode 62 (no improvement for 29 episodes)
  [Trial 00] DONE  length=430.45  time=105.6s
  [Trial 01] Starting o

[I 2026-06-26 14:05:10,300] Trial 0 finished with value: 433.75320543090805 and parameters: {'num_episodes': 75}. Best is trial 0 with value: 433.75320543090805.


  [T02] Early stopping at episode 67 (no improvement for 29 episodes)
  [Trial 02] DONE  length=434.01  time=114.3s
  [Trial 00] Starting on cpu
  [T00] Ep    1/150  best=    473.33  overall=    473.33  loss= -0.0034  ant_avg=   1000.77  ant_std=  85.14 ★
  [T00] Ep   10/150  best=    455.39  overall=    442.75  loss= -0.0030  ant_avg=    956.83  ant_std= 105.03
  [T00] Ep   20/150  best=    463.86  overall=    442.75  loss= -0.0026  ant_avg=    794.20  ant_std=  75.70
  [T00] Ep   30/150  best=    436.55  overall=    436.55  loss= -0.0009  ant_avg=    563.88  ant_std=  57.13 ★
  [T00] Ep   40/150  best=    435.82  overall=    435.82  loss= -0.0002  ant_avg=    478.65  ant_std=  33.95
  [T00] Ep   50/150  best=    435.82  overall=    435.82  loss= -0.0000  ant_avg=    443.44  ant_std=  23.69 ★
  [T00] Ep   60/150  best=    435.82  overall=    435.82  loss= -0.0000  ant_avg=    435.82  ant_std=   0.00 ★
  [T00] Early stopping at episode 70 (no improvement for 29 episodes)
  [Trial 00] D

[I 2026-06-26 14:10:51,848] Trial 1 finished with value: 433.7253583996371 and parameters: {'num_episodes': 150}. Best is trial 1 with value: 433.7253583996371.


...

  [T02] Early stopping at episode 73 (no improvement for 29 episodes)
  [Trial 02] DONE  length=433.86  time=124.1s
  [Trial 00] Starting on cpu
  [T00] Ep    1/100  best=    448.53  overall=    448.53  loss= -0.0037  ant_avg=   1008.69  ant_std=  84.22 ★
  [T00] Ep   10/100  best=    493.08  overall=    443.51  loss= -0.0035  ant_avg=    965.05  ant_std=  80.00
  [T00] Ep   20/100  best=    461.62  overall=    443.51  loss= -0.0026  ant_avg=    801.80  ant_std=  74.92
  [T00] Ep   30/100  best=    460.37  overall=    439.63  loss= -0.0010  ant_avg=    570.83  ant_std=  49.48
  [T00] Ep   40/100  best=    436.48  overall=    436.48  loss= -0.0002  ant_avg=    489.25  ant_std=  31.83 ★
  [T00] Ep   50/100  best=    434.44  overall=    434.44  loss= -0.0001  ant_avg=    438.41  ant_std=  11.94 ★
  [T00] Ep   60/100  best=    434.44  overall=    434.44  loss= -0.0000  ant_avg=    434.44  ant_std=   0.00 ★
  [T00] Ep   70/100  best=    434.44  overall=    434.44  loss= -0.0000  ant_avg=   

[I 2026-06-26 15:36:05,022] Trial 18 finished with value: 435.9623535615026 and parameters: {'num_episodes': 100}. Best is trial 15 with value: 431.0280174212566.


  [T02] Early stopping at episode 36 (no improvement for 29 episodes)
  [Trial 02] DONE  length=439.51  time=64.2s
  [Trial 00] Starting on cpu
  [T00] Ep    1/75  best=    471.61  overall=    471.61  loss= -0.0038  ant_avg=    985.01  ant_std=  93.53 ★
  [T00] Ep   10/75  best=    528.42  overall=    451.34  loss= -0.0035  ant_avg=    970.69  ant_std=  71.44
  [T00] Ep   20/75  best=    437.34  overall=    437.34  loss= -0.0030  ant_avg=    797.43  ant_std=  62.80 ★
  [T00] Ep   30/75  best=    448.66  overall=    437.34  loss= -0.0009  ant_avg=    570.93  ant_std=  58.06
  [T00] Ep   40/75  best=    431.37  overall=    431.37  loss= -0.0003  ant_avg=    440.94  ant_std=  28.07 ★
  [T00] Ep   50/75  best=    431.37  overall=    431.37  loss= -0.0000  ant_avg=    431.37  ant_std=   0.00 ★
  [T00] Ep   60/75  best=    431.37  overall=    431.37  loss= -0.0000  ant_avg=    431.37  ant_std=   0.00 ★
  [T00] Early stopping at episode 64 (no improvement for 29 episodes)
  [Trial 00] DONE  l

[I 2026-06-26 15:41:38,334] Trial 19 finished with value: 432.07569149674083 and parameters: {'num_episodes': 75}. Best is trial 15 with value: 431.0280174212566.


  [T02] Early stopping at episode 58 (no improvement for 29 episodes)
  [Trial 02] DONE  length=431.95  time=99.4s

  eil51 — Optuna complete
Best Path Length : 431.03

Best Hyperparameters:
num_episodes: 100

Running final evaluation with best hyperparameters...

  [Trial 999] Starting on cpu
  [T999] Ep    1/100  best=    473.93  overall=    473.93  loss= -0.0037  ant_avg=   1012.93  ant_std=  69.68 ★
  [T999] Ep   10/100  best=    489.12  overall=    446.98  loss= -0.0034  ant_avg=    961.03  ant_std=  80.79
  [T999] Ep   20/100  best=    482.50  overall=    435.14  loss= -0.0026  ant_avg=    770.20  ant_std=  55.56
  [T999] Ep   30/100  best=    459.48  overall=    435.14  loss= -0.0011  ant_avg=    593.72  ant_std=  47.85
  [T999] Ep   40/100  best=    431.37  overall=    431.33  loss= -0.0004  ant_avg=    499.17  ant_std=  45.74
  [T999] Ep   50/100  best=    431.37  overall=    431.33  loss=  0.0000  ant_avg=    434.29  ant_std=  13.66
  [T999] Ep   60/100  best=    431.37  over

# New Algorithm

In [1]:
"""
Hybrid ACO + RL-GNN for TSP
============================
Replaces tabular Q-learning with a Graph Attention Network (GAT) policy.

Architecture:
  - GNN encoder:  2-layer GAT over fully-connected city graph
                  -> node embeddings h_i in R^{hidden_dim}
  - Policy head:  attention between current-city query and all unvisited keys
                  -> probability distribution over next city
  - Training:     REINFORCE with baseline (mean tour length of the ant batch)
  - ACO:  pheromone matrix + 2-opt local search unchanged from original


Dependencies:
    pip install torch torch-geometric numpy
    (torch-geometric wheels: https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html)
"""

import os
import csv
import random
import math
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data


# =============================================================================
# 1.  TSP file reader
# =============================================================================

def read(instance_file):
    coords = []
    in_coord_section = False
    with open(instance_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith("NODE_COORD_SECTION"):
                in_coord_section = True
                continue
            if not in_coord_section:
                continue
            if line.startswith("EOF"):
                break
            parts = line.split()
            if len(parts) < 3:
                continue
            _, x, y = parts[:3]
            coords.append([float(x), float(y)])
    return np.array(coords)


# =============================================================================
# 2.  Distance / path utilities
# =============================================================================

def calculate_path_length(path, distances):
    return sum(distances[path[i]][path[i + 1]] for i in range(len(path) - 1))


def local_search_2opt(path, distances, num_cities):
    best = path[:]
    best_len = calculate_path_length(best, distances)
    improved = True
    while improved:
        improved = False
        for i in range(1, num_cities - 2):
            for j in range(i + 1, num_cities):
                if j - i == 1:
                    continue
                candidate = path[:i] + path[i:j][::-1] + path[j:]
                clen = calculate_path_length(candidate, distances)
                if clen < best_len:
                    best, best_len = candidate, clen
                    improved = True
        path = best
    return best


# =============================================================================
# 3.  GNN Policy Network  (Graph Attention Network backbone)
# =============================================================================

class GNNPolicyNet(nn.Module):
    """
    Graph Attention Network that maps city coordinates to a next-city
    probability distribution conditioned on the current city.

    Input features per node (dim=5):
        [x_norm, y_norm, dist_to_centroid_norm, sin(angle), cos(angle)]

    Architecture:
        Linear(5 -> hidden_dim)
        GATConv(hidden_dim -> hidden_dim, heads=num_heads, concat) x num_layers
        Policy head: score(i) = (W_q * h_current) . (W_k * h_i) / sqrt(D)
    """

    def __init__(self, hidden_dim: int = 128, num_heads: int = 4, num_layers: int = 2):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Node feature encoder
        self.input_proj = nn.Linear(5, hidden_dim)

        # GAT layers: each head outputs hidden_dim//num_heads, concat -> hidden_dim
        head_dim = hidden_dim // num_heads
        self.gat_layers = nn.ModuleList([
            GATConv(hidden_dim, head_dim, heads=num_heads, concat=True, dropout=0.0)
            for _ in range(num_layers)
        ])

        # Attention-based policy head
        self.W_query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_key   = nn.Linear(hidden_dim, hidden_dim, bias=False)

        # Xavier initialisation
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def encode(self, data: Data) -> torch.Tensor:
        """Run GAT message passing, return node embeddings [N, hidden_dim]."""
        h = F.relu(self.input_proj(data.x))
        for gat in self.gat_layers:
            h = F.elu(gat(h, data.edge_index))
        return h

    def policy(
        self,
        node_emb: torch.Tensor,     # [N, hidden_dim]
        current_idx: int,
        visited_mask: torch.Tensor  # bool [N], True = already visited
    ) -> torch.Tensor:
        """Return log-probabilities over cities; visited cities -> -inf."""
        query  = self.W_query(node_emb[current_idx]).unsqueeze(0)  # [1, D]
        keys   = self.W_key(node_emb)                              # [N, D]
        scale  = math.sqrt(self.hidden_dim)
        logits = (query @ keys.T).squeeze(0) / scale               # [N]
        logits = logits.masked_fill(visited_mask, float('-inf'))
        return F.log_softmax(logits, dim=-1)

    def forward(self, data, current_idx, visited_mask):
        return self.policy(self.encode(data), current_idx, visited_mask)


# =============================================================================
# 4.  Graph construction helper
# =============================================================================

def build_graph(coords: np.ndarray, device: torch.device) -> Data:
    """
    Fully-connected PyG Data object from city coordinates.
    Node features: [x_norm, y_norm, dist_centroid_norm, sin_theta, cos_theta]
    """
    N = len(coords)

    lo, hi = coords.min(0), coords.max(0)
    span = (hi - lo).clip(min=1e-6)
    coords_n = (coords - lo) / span

    centroid = coords_n.mean(0)
    diffs    = coords_n - centroid
    dist_c   = np.linalg.norm(diffs, axis=1, keepdims=True)
    max_d    = dist_c.max() + 1e-6
    angles   = np.arctan2(diffs[:, 1], diffs[:, 0])

    node_feats = np.concatenate([
        coords_n,
        dist_c / max_d,
        np.sin(angles)[:, None],
        np.cos(angles)[:, None],
    ], axis=1).astype(np.float32)

    # Fully-connected edges (no self-loops)
    src = [i for i in range(N) for j in range(N) if i != j]
    dst = [j for i in range(N) for j in range(N) if i != j]

    return Data(
        x          = torch.tensor(node_feats, device=device),
        edge_index = torch.tensor([src, dst], dtype=torch.long, device=device),
    )


# =============================================================================
# 5.  ACO pheromone update
# =============================================================================

def update_pheromones(pheromones, paths, distances, w_reward, rho, num_cities):
    delta = np.zeros((num_cities, num_cities))
    for path in paths:
        reward = w_reward / calculate_path_length(path, distances)
        for i in range(len(path) - 1):
            delta[path[i]][path[i + 1]] += reward
    pheromones += -rho * pheromones + delta


# =============================================================================
# 6.  Hybrid ACO + RL-GNN main loop
# =============================================================================

def hybrid_aco_rl_gnn(
    coords: np.ndarray,
    distances: np.ndarray,
    policy_net: GNNPolicyNet,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    # ACO params
    num_episodes: int  = 100,
    num_ants: int      = 20,
    rho: float         = 0.3,
    w_reward: float    = 10.0,
    delta: float       = 3.0,
    beta: float        = 3.0,
    # 2-opt
    num_local_search_iterations: int = 25,
    # RL params
    entropy_coef: float = 0.01,
    # GNN vs pheromone blend  (0 = pure ACO, 1 = pure GNN)
    gnn_weight: float  = 0.6,
    # Logging
    trial_id: int      = 0,
    print_every: int   = 10,
):
    """
    Each episode:
      1. Encode graph with GAT -> node embeddings (no grad, rollout only).
      2. Each ant constructs a tour by blending GNN log-probs + ACO desirability.
      3. Recompute log-probs WITH grad, run REINFORCE + entropy regularisation.
      4. 2-opt refinement on best ant tour.
      5. Pheromone update on best tour.
    """
    num_cities = len(coords)
    graph_data = build_graph(coords, device)
    pheromones = np.ones((num_cities, num_cities))

    overall_best_path = None
    overall_best_len  = float('inf')

    #early_patience critertion
    patience = 20
    episodes_without_improvement = 0
    
    for episode in range(num_episodes):
        policy_net.train()

        # -- Encode graph once per episode (no grad needed for rollout) -------
        with torch.no_grad():
            node_emb = policy_net.encode(graph_data)  # [N, D]

        paths         = []
        entropy_sums  = []

        # -- Ant rollouts ------------------------------------------------------
        for ant in range(num_ants):
            start_city = random.randint(0, num_cities - 1)
            path       = [start_city]
            visited    = torch.zeros(num_cities, dtype=torch.bool, device=device)
            visited[start_city] = True

            ant_entropies = []

            for step in range(num_cities - 1):
                current = path[-1]

                # GNN log-probabilities
                with torch.no_grad():
                    log_probs_gnn = policy_net.policy(node_emb, current, visited)

                # ACO desirability scores (pheromone x heuristic)
                aco_scores = np.zeros(num_cities)
                for j in range(num_cities):
                    if not visited[j].item():
                        aco_scores[j] = (
                            pheromones[current][j] ** delta *
                            (1.0 / (distances[current][j] + 1e-12)) ** beta
                        )
                aco_sum = aco_scores.sum()
                if aco_sum > 0:
                    aco_scores /= aco_sum

                aco_log = torch.tensor(
                    np.log(aco_scores + 1e-12), dtype=torch.float32, device=device
                ).masked_fill(visited, float('-inf'))

                # Blend in log space and sample
                combined = (
                    gnn_weight * log_probs_gnn +
                    (1 - gnn_weight) * F.log_softmax(aco_log, dim=-1)
                ).masked_fill(visited, float('-inf'))

                probs     = F.softmax(combined, dim=-1)
                dist_obj  = torch.distributions.Categorical(probs=probs)
                next_city = dist_obj.sample().item()

                ant_entropies.append(dist_obj.entropy())
                path.append(next_city)
                visited[next_city] = True

            path.append(path[0])  # close tour
            paths.append(path)
            entropy_sums.append(torch.stack(ant_entropies).mean())

        # -- Tour lengths ------------------------------------------------------
        tour_lengths = np.array([calculate_path_length(p, distances) for p in paths])

        # -- REINFORCE with mean baseline -------------------------------------
        baseline = tour_lengths.mean()
        advantages = torch.tensor(
            -(tour_lengths - baseline), dtype=torch.float32, device=device
        )

        policy_loss = torch.zeros(1, device=device).squeeze()  # scalar, grad-safe initializer
        
        for i in range(num_ants):
            log_probs_list = []
            visited_set = {paths[i][0]}               # plain Python set — no autograd involvement
            node_emb_bp = policy_net.encode(graph_data)  # fresh encoding WITH grad
        
            for step in range(num_cities - 1):
                cur = paths[i][step]
                nxt = paths[i][step + 1]
        
                # Build a brand-new mask tensor each step — never mutates a tracked tensor
                mask = torch.zeros(num_cities, dtype=torch.bool, device=device)
                for v in visited_set:
                    mask[v] = True
        
                lp = policy_net.policy(node_emb_bp, cur, mask)
                log_probs_list.append(lp[nxt])
                visited_set.add(nxt)                  # update set, not any tensor
        
            policy_loss = policy_loss - advantages[i] * torch.stack(log_probs_list).sum()
        
        policy_loss  = policy_loss / num_ants
        entropy_loss = -entropy_coef * torch.stack(entropy_sums).mean()
        total_loss   = policy_loss + entropy_loss

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_net.parameters(), max_norm=1.0)
        optimizer.step()

        # -- 2-opt on best ant ------------------------------------------------
        best_idx  = int(np.argmin(tour_lengths))
        best_path = paths[best_idx]
        best_len  = tour_lengths[best_idx]

        for _ in range(num_local_search_iterations):
            refined     = local_search_2opt(best_path, distances, num_cities)
            refined_len = calculate_path_length(refined, distances)
            if refined_len < best_len:
                best_path, best_len = refined, refined_len

        if best_len < overall_best_len:
            overall_best_len = best_len
            overall_best_path = best_path[:]
            episodes_without_improvement = 0
        else:
            episodes_without_improvement += 1

        # -- Pheromone update -------------------------------------------------
        update_pheromones(pheromones, [best_path], distances, w_reward, rho, num_cities)

        # Early stopping
        if episodes_without_improvement >= patience:
            print(
                f"  [T{trial_id:02d}] Early stopping at episode "
                f"{episode + 1} (no improvement for {patience} episodes)"
            )
            break

        # -- Episode progress print -------------------------------------------
        freq = max(print_every, 1)
        if (episode + 1) % freq == 0 or episode == 0:
            star = " ★" if best_len == overall_best_len else ""
            print(
                f"  [T{trial_id:02d}] Ep {episode+1:4d}/{num_episodes}"
                f"  best={best_len:10.2f}"
                f"  overall={overall_best_len:10.2f}"
                f"  loss={total_loss.item():8.4f}"
                f"  ant_avg={tour_lengths.mean():10.2f}"
                f"  ant_std={tour_lengths.std():7.2f}"
                f"{star}",
                flush=True,
            )

    return overall_best_path


# =============================================================================
# 7.  Trial runner
# =============================================================================

def run_trial(trial, coords, distances, params):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    policy_net = GNNPolicyNet(
        hidden_dim = params["hidden_dim"],
        num_heads  = params["num_heads"],
        num_layers = params["num_layers"],
    ).to(device)

    optimizer = torch.optim.Adam(
        policy_net.parameters(),
        lr           = params["lr"],
        weight_decay = 1e-5,
    )

    print(f"  [Trial {trial:02d}] Starting on {device}")
    start = datetime.now()

    best_path = hybrid_aco_rl_gnn(
        coords       = coords,
        distances    = distances,
        policy_net   = policy_net,
        optimizer    = optimizer,
        device       = device,
        num_episodes = params["num_episodes"],
        num_ants     = params["num_ants"],
        rho          = params["rho"],
        w_reward     = params["w_reward"],
        delta        = params["delta"],
        beta         = params["beta"],
        num_local_search_iterations = params["num_local_search_iterations"],
        entropy_coef = params["entropy_coef"],
        gnn_weight   = params["gnn_weight"],
        trial_id     = trial,
        print_every  = params["print_every"],
    )

    length  = calculate_path_length(best_path, distances)
    elapsed = (datetime.now() - start).total_seconds()
    print(f"  [Trial {trial:02d}] DONE  length={length:.2f}  time={elapsed:.1f}s")
    return trial, length, elapsed


# =============================================================================
# 8.  Hyperparameters
# =============================================================================

global_params = {
    # GNN architecture
    "hidden_dim": 128,
    "num_heads": 2,
    "num_layers": 4,
    "lr": 0.000540030125825684,

    # ACO
    "num_ants": 37,
    "num_episodes": 100,
    "rho": 0.17848728876955464,
    "w_reward": 10.0,
    "delta": 2.2721263607571762,
    "beta": 2.9664340523909023,

    # 2-opt
    "num_local_search_iterations": 19,

    # Early stopping
    "patience": 29,

    # RL
    "entropy_coef": 0.001559946051309167,

    # Blend (0 = pure ACO heuristic, 1 = pure GNN policy)
    "gnn_weight": 0.3267091672644127,

    # Logging
    "print_every": 10,
}




instances_dir  = "SmallMedium"
instance_files = [
    os.path.join(instances_dir, f)
    for f in os.listdir(instances_dir)
    if f.endswith(".tsp")
]

os.makedirs("Results", exist_ok=True)
csv_file   = "Results/ACO-RLGNN-2OptAlgorithm.csv"
num_trials = 5
algorithm  = "ACO-RLGNN-2OptAlgorithm"

with open(csv_file, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Instance", "Algorithm", "Trial", "PathLength", "TimeSeconds"])

    for instance_file in instance_files:
        instance_name = os.path.basename(instance_file).replace(".tsp", "")

        print(f"\n{'='*52}")
        print(f"  Instance : {instance_name}")
        print(f"  Trials   : {num_trials}  |  "
              f"Episodes : {global_params['num_episodes']}  |  "
              f"Ants : {global_params['num_ants']}")
        print(f"{'='*52}")

        coords     = read(instance_file)
        num_cities = len(coords)
        distances  = np.sqrt(
            (coords[:, None, 0] - coords[None, :, 0]) ** 2 +
            (coords[:, None, 1] - coords[None, :, 1]) ** 2
        )

        results = []

        for trial in range(1, num_trials + 1):
            print(f"\n  -- Trial {trial}/{num_trials} " + "-" * 36)
            trial_num, length, tsec = run_trial(trial, coords, distances, global_params)
            results.append((trial_num, length, tsec))
            writer.writerow([instance_name, algorithm, trial_num, length, tsec])
            f.flush()  # save after every trial so partial results are safe

        lengths = [r[1] for r in results]
        times   = [r[2] for r in results]
        print(f"\n{'='*52}")
        print(f"  {instance_name} — all trials complete")
        print(f"  Best  : {min(lengths):.2f}")
        print(f"  Mean  : {sum(lengths)/len(lengths):.2f}")
        print(f"  Worst : {max(lengths):.2f}")
        print(f"  AvgTime: {sum(times)/len(times):.1f}s"
              f"  (min={min(times):.1f}s  max={max(times):.1f}s)")
        print(f"{'='*52}")

print(f"\nAll done. Results saved to: {csv_file}")


  Instance : bier127
  Trials   : 5  |  Episodes : 100  |  Ants : 37

  -- Trial 1/5 ------------------------------------
  [Trial 01] Starting on cpu
  [T01] Ep    1/100  best= 138746.77  overall= 138746.77  loss= -0.1526  ant_avg= 305572.47  ant_std=21482.11 ★
  [T01] Ep   10/100  best= 130542.65  overall= 127246.26  loss= -0.0310  ant_avg= 307982.58  ant_std=24044.19
  [T01] Ep   20/100  best= 131038.91  overall= 125674.60  loss= -0.0580  ant_avg= 309879.78  ant_std=20451.35
  [T01] Ep   30/100  best= 135142.61  overall= 125674.60  loss=  0.0231  ant_avg= 304455.63  ant_std=17285.30
  [T01] Early stopping at episode 39 (no improvement for 20 episodes)
  [Trial 01] DONE  length=125674.60  time=804.9s

  -- Trial 2/5 ------------------------------------
  [Trial 02] Starting on cpu
  [T02] Ep    1/100  best= 126870.89  overall= 126870.89  loss= -0.0310  ant_avg= 311215.16  ant_std=19398.68 ★
  [T02] Ep   10/100  best= 131895.19  overall= 123022.65  loss= -0.0343  ant_avg= 304576.83  

KeyboardInterrupt: 